# Linear Regression on Medical Data for prediction

Goal: Fit and evaluate linear regression on a medical dataset and observe the effect of correlated predictors.

In [ ]:
import os
import statsmodels.api as sm
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, r2_score



def load_dataset(name: str) -> pd.DataFrame:
    base_url = (
        "https://raw.githubusercontent.com/"
        "JRandy77/QLS-MiCM_Introduction_to_supervised_machine_learning/main/Exercises/data/"
    )
    url = f"{base_url}{name}.csv"
    
    try:
        return pd.read_csv(url)
    except Exception as e:
        raise FileNotFoundError(f"Could not load dataset '{name}' from {url}") from e



def basic_train_test_split(df: pd.DataFrame, target: str, test_size: float = 0.2, random_state: int = 42):
    X = df.drop(columns=[target])
    y = df[target]
    return train_test_split(X, y, test_size=test_size, random_state=random_state)


def build_preprocessor(num_features, cat_features):
    numeric_transformer = StandardScaler()
    categorical_transformer = OneHotEncoder(handle_unknown="ignore")
    preprocessor = ColumnTransformer(
        transformers=[
            ("num", numeric_transformer, num_features),
            ("cat", categorical_transformer, cat_features),
        ]
    )
    return preprocessor



In [ ]:


df = load_dataset("bp")
X_train, X_test, y_train, y_test = basic_train_test_split(df, target="sbp")


In [ ]:
# TODO 1.1: Fit a simple linear regression using age only
# Tutorial: https://scikit-learn.org/stable/modules/linear_model.html#ordinary-least-squares
model_age = LinearRegression()
model_age.fit(X_train[["age"]], y_train)

# Evaluate predictive performance
y_pred_age = model_age.predict(X_test[["age"]])
print("MAE (age only):", mean_absolute_error(y_test, y_pred_age))
print("R2 (age only):", r2_score(y_test, y_pred_age))

# Inspect coefficient (effect size)
print("Intercept:", model_age.intercept_)
print("Coefficient for age:", model_age.coef_[0])


In [ ]:
# ***EXERCISE*** 1.2: Fit a model with multiple predictors
features = ["age", "bmi", "sodium"]
model_multi = LinearRegression()
#fit a linear regression on the data with the specified features

y_pred_multi = #predictions on test set

print("MAE (multi):", mean_absolute_error(y_test, y_pred_multi))
print("R2 (multi):", r2_score(y_test, y_pred_multi))

print("\nCoefficients (effect sizes):")
for f, c in zip(features, model_multi.coef_):
    print(f"{f:10s} {c: .3f}")


In [ ]:
# ***EXERCISE*** 1.3: Add correlated feature (weight) and refit
features_corr = []
model_corr = LinearRegression()
model_corr.fit(X_train[features_corr], y_train)

print("Coefficients with correlated predictor:")
for f, c in zip(features_corr, model_corr.coef_):
    print(f"{f:10s} {c: .3f}")


In [ ]:
# BONUS: Inspect feature correlations
print("\nFeature correlations:")
print(df[features_corr + ["sbp"]].corr(numeric_only=True))


In [ ]:

X = sm.add_constant(X_train[["age", "bmi", "sodium"]])
ols = sm.OLS(y_train, X).fit()

# Extract key information
results = pd.DataFrame({
    "coef": ols.params,
    "std_err": ols.bse,
    "t_value": ols.tvalues,
    "p_value": ols.pvalues
})

# Display neatly
print(results.round(4))


In [ ]:


# Create interaction term manually
df["bmi_sex_interaction"] = df["bmi"] * df["sex"]

X = df[["age", "bmi", "sodium", "exercise_minutes", "sex", "bmi_sex_interaction"]]
y = df["sbp"]

# Fit OLS model
X_ = sm.add_constant(X)
ols = sm.OLS(y, X_).fit()

# Compact output
pd.DataFrame({
    "coef": ols.params,
    "p_value": ols.pvalues
}).round(4)


### Reflection:
1. Which predictors have large effect sizes but low p-values (statistically significant)?
2. Which predictors are correlated with each other?
3. Did adding more predictors improve R^2 or MAE meaningfully?
4. If not, what might that tell us about model complexity?
